# TrueState - Data Synthesis & Bias Exploration

This notebook is intended to synthesize datasets that clearly demonstrate two major clinical AI biases:
1. **Observation Policy Bias (IV-OPIL):** Patients in the ICU are measured more frequently (e.g. q1h) than general ward patients (e.g. q4h). A standard AI model overestimates risk for highly measured patients.
2. **Selective Label Bias (Proxy-SLCD):** Marginalized patients often do not receive expensive/rare diagnostic tests, making them "invisible" to standard supervised learning.

The generated data here mirrors what is seeded in our PostgreSQL database for the Express/Next.js dashboard to consume.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries imported successfully!")

In [ ]:
# 1. Observation Policy Bias (IV-OPIL)
# We generate 2 patients with the SAME underlying condition (elevated heart rate).
# Patient A is in the ICU (measured every hour). Patient B is in the General Ward (measured every 4 hours).

time_icu = np.arange(0, 24, 1)
time_ward = np.arange(0, 24, 4)

# True underlying state is similar
true_hr_icu = 100 + 5 * np.sin(time_icu / 3) + np.random.normal(0, 2, len(time_icu))
true_hr_ward = 100 + 5 * np.sin(time_ward / 3) + np.random.normal(0, 2, len(time_ward))

# Standard AI naive risk score often correlates highly with simply the *number* of abnormal readings.
risk_icu = (true_hr_icu > 100).cumsum() / len(time_icu)
risk_ward = (true_hr_ward > 100).cumsum() / len(time_ward)

plt.figure(figsize=(12, 5))

# Plot HR
plt.subplot(1, 2, 1)
plt.plot(time_icu, true_hr_icu, 'ro-', label='Patient A (ICU - High Freq)')
plt.plot(time_ward, true_hr_ward, 'bo-', label='Patient B (Ward - Low Freq)')
plt.title("Heart Rate Over 24 Hours")
plt.xlabel("Hours")
plt.ylabel("Heart Rate")
plt.legend()

# Plot Naive Risk
plt.subplot(1, 2, 2)
plt.plot(time_icu, risk_icu, 'r--', label='Patient A - Standard AI Risk')
plt.plot(time_ward, risk_ward, 'b--', label='Patient B - Standard AI Risk')
plt.title("Standard AI Diverging Risk (Biased)")
plt.xlabel("Hours")
plt.ylabel("Calculated Risk Score")
plt.legend()

plt.tight_layout()
plt.show()